In [5]:
import pandas as pd
import plotly.express as px
from pathlib import Path

fail = Path("rfm_results_clean.csv")
rfm = pd.read_csv("rfm_results_clean.csv")
print("RFM tabeli mõõtmed:", rfm.shape)
print(rfm.head())

RFM tabeli mõõtmed: (2540, 9)
   customer_id  recency_days  frequency  monetary  R_score  F_score  M_score  \
0       2001.0            91          2    203.92        4        1        1   
1       2004.0            71          2   1198.56        4        1        4   
2       2005.0           148          4    959.60        3        4        4   
3       2006.0           477          1    327.06        1        1        1   
4       2007.0            29          1    318.63        5        1        1   

   RFM_score    segment  
0          6    At risk  
1          9  Potential  
2         11      Loyal  
3          3       Lost  
4          7  Potential  


In [19]:
# Diagramm 1. Segmentide veerg
if "Segment" in rfm.columns:
    segment_col = "Segment"
elif "segment" in rfm.columns:
    segment_col = "segment"
else:
    raise ValueError(
        f"Segmendi veerg puudub. Faili veerud: {rfm.columns.tolist()}"
    )

# järjekord
segmentide_järjekord = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

rfm[segment_col] = rfm[segment_col].replace({
    "At risk": "At Risk"
})

# klientide arv segmentides
segmentide_arv = (
    rfm.groupby(segment_col, observed=True)["customer_id"]
    .nunique()
    .reset_index(name="klientide_arv")
)

# Teen tulpdiagrammi
fig = px.bar(
    segmentide_arv,
    x=segment_col,
    y="klientide_arv",
    color=segment_col,
    text="klientide_arv",
    title="UrbanStyle: klientide arv segmentide kaupa",
    labels={
        segment_col: "Kliendisegment",
        "klientide_arv": "Klientide arv"
    },
    category_orders={segment_col: segmentide_järjekord},
    template="plotly_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.update_yaxes(rangemode="tozero")
fig.show()

In [20]:
# Diagramm 2 -- scatter
rfm["segment"] = rfm["segment"].replace({"At risk": "At Risk"})
segmentide_järjekord = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

fig = px.scatter(
    rfm,
    x="recency_days",
    y="monetary",
    color="segment",
    size="frequency",
    hover_name="customer_id",
    size_max=40,
    opacity=0.7,
    title="UrbanStyle: ostude hiljutisus ja klientide kogukulutus",
    labels={
        "recency_days": "Päevi viimasest ostust",
        "monetary": "Kogukulutus (EUR)",
        "frequency": "Ostude arv",
        "segment": "Kliendisegment",
        "customer_id": "Kliendi ID"
    },
    category_orders={"segment": segmentide_järjekord},
    template="plotly_white"
)

fig.show()

In [11]:
# Diagramm 3. 10 suurima kogukulutusega VIP-klienti

top_vip = (
    rfm[rfm["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
    .copy()
)

# Kliendi ID peab diagrammil olema kategooria
top_vip["customer_id"] = top_vip["customer_id"].astype(str)

if top_vip.empty:
    print("VIP Champions segmendis pole kliente.")
else:
    fig = px.bar(
        top_vip,
        x="monetary",
        y="customer_id",
        orientation="h",
        text="monetary",
        title="UrbanStyle: top 10 VIP-klienti kogukulutuse järgi",
        labels={
            "monetary": "Kogukulutus (EUR)",
            "customer_id": "Kliendi ID"
        },
        template="plotly_white"
    )

    fig.update_yaxes(
        type="category",
        categoryorder="array",
        categoryarray=top_vip["customer_id"].tolist(),
        autorange="reversed"
    )

    fig.update_traces(
        texttemplate="%{x:,.2f} €",
        textposition="auto"
    )

    fig.show()

In [13]:
import pandas as pd

rfm = pd.read_csv("rfm_results_clean.csv")

vip = rfm[rfm["segment"] == "VIP Champions"]
at_risk = rfm[rfm["segment"] == "At Risk"]

kogukaive = rfm["monetary"].sum()
vip_kaive = vip["monetary"].sum()
vip_osakaal = vip_kaive / kogukaive * 100 if kogukaive > 0 else 0

print(
    f"Marko, analüüsitud klientidest {vip['customer_id'].nunique()} "
    f"kuulub VIP Champions segmenti. "
    f"Nende kogukulutus on {vip_kaive:,.2f} eurot, mis moodustab "
    f"{vip_osakaal:.1f}% analüüsitud klientide kogukulutusest. "
    f"At Risk segmendis on {at_risk['customer_id'].nunique()} klienti; "
    f"see viitab tähelepanu vajavale kliendirühmale, kuid ei tõenda nende lahkumist. "
    f"VIP-klientidele soovitan varajast ligipääsu uutele kollektsioonidele "
    f"ja eritingimusi ning At Risk klientidele personaalset "
    f"14-päevast tagasivõitmispakkumist, mille tulemust mõõta "
    f"30 päeva kordusostumäära ja kontrollrühmaga võrdlemise kaudu. "
    f"Kuna RFM-analüüsi viitekuupäev on 28.02.2025, tuleb enne "
    f"kampaaniate käivitamist kliendisegmendid värskendada."
)

Marko, analüüsitud klientidest 453 kuulub VIP Champions segmenti. Nende kogukulutus on 1,141,766.71 eurot, mis moodustab 42.8% analüüsitud klientide kogukulutusest. At Risk segmendis on 0 klienti; see viitab tähelepanu vajavale kliendirühmale, kuid ei tõenda nende lahkumist. VIP-klientidele soovitan varajast ligipääsu uutele kollektsioonidele ja eritingimusi ning At Risk klientidele personaalset 14-päevast tagasivõitmispakkumist, mille tulemust mõõta 30 päeva kordusostumäära ja kontrollrühmaga võrdlemise kaudu. Kuna RFM-analüüsi viitekuupäev on 28.02.2025, tuleb enne kampaaniate käivitamist kliendisegmendid värskendada.
